In [1]:
# Variational Autoencoder (VAE)

import numpy as np
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

# 🔹 Parameters
input_dim = 20
latent_dim = 2

# 🔹 Encoder
inputs = Input(shape=(input_dim,))
encoded = Dense(10, activation='relu')(inputs)
latent = Dense(latent_dim)(encoded)

# 🔹 Decoder
decoded = Dense(10, activation='relu')(latent)
outputs = Dense(input_dim, activation='sigmoid')(decoded)

# 🔹 VAE Model
vae = Model(inputs, outputs)
vae.compile(optimizer='adam', loss='mse')

# 🔹 Dummy Data
X_train = np.random.rand(1000, input_dim)

# 🔹 Train
vae.fit(X_train, X_train, epochs=10, batch_size=32)

# 🔹 Test Reconstruction
sample = np.random.rand(1, input_dim)
reconstructed = vae.predict(sample)

print("Original:", sample)
print("Reconstructed:", reconstructed)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0822
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0816
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0806
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0795
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0786
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0778
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0772
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0766
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0761
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0757
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
Original: [[0.87901644 0.89125763 0.57306022 0.07062532 0.2927741  0.66497933
  0.93470726 0.2639611  0.74097402 0.80961225 0.64416181 0.95108388
  0.19835076 0.47487175 0.1767744  0.17447286 0.64793568 0.71169038
  0.4167919  0.83548676]]
Reconstructed: [[0.44061288 0.5697382  0.34069297 0.50172687 0.5643955

In [4]:
# Generative Adversarial Network (GAN)

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# 🔹 1. Generator Model
generator = Sequential([
    Dense(16, activation='relu', input_shape=(10,)),
    Dense(1, activation='sigmoid')
])

# 🔹 2. Discriminator Model
discriminator = Sequential([
    Dense(16, activation='relu', input_shape=(1,)),
    Dense(1, activation='sigmoid')
])

discriminator.compile(optimizer=Adam(learning_rate=0.001),
                      loss='binary_crossentropy',
                      metrics=['accuracy'])

# 🔹 3. Combined GAN Model
discriminator.trainable = False
gan = Sequential([generator, discriminator])

gan.compile(optimizer=Adam(learning_rate=0.001),
            loss='binary_crossentropy')

# 🔹 4. Training Loop
epochs = 200
batch_size = 32

for epoch in range(epochs):

    # -------- Train Discriminator --------
    real_data = np.random.rand(batch_size, 1)

    noise = np.random.rand(batch_size, 10)
    fake_data = generator(noise, training=False)   # faster than predict

    X = np.vstack((real_data, fake_data))
    y = np.vstack((np.ones((batch_size, 1)),
                   np.zeros((batch_size, 1))))

    discriminator.trainable = True
    d_loss, d_acc = discriminator.train_on_batch(X, y)

    # -------- Train Generator --------
    noise = np.random.rand(batch_size, 10)
    misleading_labels = np.ones((batch_size, 1))  # trick discriminator

    discriminator.trainable = False
    g_loss = gan.train_on_batch(noise, misleading_labels)

    # -------- Print Progress --------
    if epoch % 20 == 0:
        print(f"Epoch {epoch} | D Loss: {d_loss:.4f}, Acc: {d_acc:.4f} | G Loss: {g_loss:.4f}")

# 🔹 5. Generate New Data
noise = np.random.rand(1, 10)
generated_data = generator(noise, training=False)

print("\nGenerated Data:", generated_data)

Epoch 0 | D Loss: 0.6868, Acc: 0.5000 | G Loss: 0.8164
Epoch 20 | D Loss: 0.6905, Acc: 0.5327 | G Loss: 0.7937
Epoch 40 | D Loss: 0.6938, Acc: 0.5644 | G Loss: 0.7686
Epoch 60 | D Loss: 0.6964, Acc: 0.5579 | G Loss: 0.7464
Epoch 80 | D Loss: 0.6975, Acc: 0.4863 | G Loss: 0.7299
Epoch 100 | D Loss: 0.6974, Acc: 0.4723 | G Loss: 0.7189
Epoch 120 | D Loss: 0.6961, Acc: 0.4769 | G Loss: 0.7120
Epoch 140 | D Loss: 0.6942, Acc: 0.4768 | G Loss: 0.7079
Epoch 160 | D Loss: 0.6928, Acc: 0.4712 | G Loss: 0.7051
Epoch 180 | D Loss: 0.6921, Acc: 0.4656 | G Loss: 0.7031

Generated Data: tf.Tensor([[0.61129665]], shape=(1, 1), dtype=float32)


In [5]:
# Graph Convolutional Network

import torch
import torch.nn as nn
import torch.optim as optim

# 🔹 GCN Model
class SimpleGCN(nn.Module):
    def __init__(self):
        super(SimpleGCN, self).__init__()
        self.fc = nn.Linear(10, 2)

    def forward(self, x, adj):
        x = torch.matmul(adj, x)  # aggregate neighbors
        x = self.fc(x)
        return x

# 🔹 Dummy Graph Data
num_nodes = 5
features = torch.rand(num_nodes, 10)

# Adjacency matrix (example)
adj = torch.tensor([
    [1,1,0,0,0],
    [1,1,1,0,0],
    [0,1,1,1,0],
    [0,0,1,1,1],
    [0,0,0,1,1]
], dtype=torch.float32)

labels = torch.tensor([0,1,0,1,0])

# 🔹 Model + Loss
model = SimpleGCN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 🔹 Training
for epoch in range(100):
    optimizer.zero_grad()
    output = model(features, adj)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()

print("Output:\n", output)

Output:
 tensor([[ 0.2664, -1.1619],
        [-1.1288, -0.2617],
        [-0.2522, -1.1925],
        [-1.1407, -0.0190],
        [ 0.2545, -0.9192]], grad_fn=<AddmmBackward0>)
